\begin{align}
    % --- Soft detection and reconstructability ---
    s_{ij} &= \sigma\!\left(\tau_{\text{layout}}\,(c_{ij} - \lambda_{\text{layout}})\right) \\
    n_i    &= \sum_{j} s_{ij} \\
    r_i    &= \sigma\!\left(\tau_{\text{recon}}\,(n_i - \lambda_{\text{recon}})\right) \\[10pt]
    % --- Reconstructability utility ---
    \mathcal{U}_{\text{PR}} &= \sqrt{\sum_{i} r_i + \epsilon} \\[6pt]
    % --- Energy utility ---
    \mathcal{U}_{E} &= \frac{1}{B}\sum_{i}
        \frac{r_i}{\left(\log_{10}\hat{E}_i - \log_{10} E_i\right)^2 + \delta_E} \\[6pt]
    % --- Angular utility (theta and phi) ---
    \mathcal{U}_{\text{angle}} &= \frac{1}{B}\sum_{i}
        \frac{r_i}{\left(\hat{\alpha}_i - \alpha_i\right)^2 + \delta_{\alpha}} \\[10pt]
    % --- Composite utility ---
    \mathcal{U} &= \frac{
        w_{\theta}\,\mathcal{U}_{\theta}
      + w_{\phi}\,\mathcal{U}_{\phi}
      + w_{E}\,\mathcal{U}_{E}
    }{w_{\text{div}}}
\end{align}

\begin{align}
    r_i &= \sigma
                \Bigl(
                    \sum_{j} \sigma\!\left(
                        \log_{10}\hat{E}_i - \lambda_{\text{layout}}
                    \right) - \lambda_{\text{recon}}
                \Bigr)\\[6pt]
    \mathcal{U}_{E}          &= \frac{1}{B}\sum_{i}
                                \frac{r_i}{\left(\log_{10}\hat{E}_i - \log_{10} E_i\right)^2} \\[6pt]
    \mathcal{U}_{\theta}     &= \frac{1}{B}\sum_{i}
                                \frac{r_i}{\left(\hat{\theta}_i - \theta_i\right)^2 } \\[6pt]
    \mathcal{U}_{\phi}       &= \frac{1}{B}\sum_{i}
                                \frac{r_i}{\left(\hat{\phi}_i - \phi_i\right)^2 } \\[6pt]
    \mathcal{U}              &= \frac{w_{\theta}\,\mathcal{U}_{\theta}
                                    + w_{\phi}\,\mathcal{U}_{\phi}
                                    + w_{E}\,\mathcal{U}_{E}}{w_{\text{div}}}
\end{align}

What the inputs are

- var_E[g] = the within-group variance of the (log-transformed) label across the M independent showers, for each group g = (primary p, layout s, detector i). By construction this is a
Monte-Carlo estimate of the conditional variance Var(E | q, xy) — fixed primary, fixed layout, only the shower realization varies.
- e_std = the corpus marginal std of the same log-transformed label = exactly the σ the trainer z-scores by.

Why mean(var)/σ² is the floor

The trainer's reported val loss is z-scored MSE:
$$\text{val}E=\frac{1}{N}\sum\text{tokens}\Big(\frac{\hat y - y}{\sigma_E}\Big)^2.$$
The best possible predictor is the conditional mean μ(q,xy)=E[y|q,xy]; no function of (q,xy) beats it in MSE. Its error is
$$\text{MSE}^\star=\frac{1}{N}\sum_\text{tokens}\frac{\mathbb{E}[(y-\mu)^2]}{\sigma_E^2}=\frac{\mathbb{E}{q,xy}\big[\operatorname{Var}(y\mid q,xy)\big]}{\sigma_E^2}.$$
That right-hand side is literally var_E.mean() / e_std**2. So _floor returns the Bayes-optimal z-MSE — a hard lower bound on the val loss, in the same units. Equivalently, by the law of
total variance σ² = E[Var(y|x)] + Var(E[y|x]),
$$\text{floor} = \frac{\mathbb{E}[\operatorname{Var}(y\mid x)]}{\sigma^2} = 1-\frac{\operatorname{Var}(\mu)}{\sigma^2}=1-R^2{\max}.$$
So floor ∈ [0,1], floor=1 ⇔ inputs carry no info, floor=0 ⇔ deterministic. The measured 0.306/0.414 sit in [0,1] as they must.

floor_total = 0.5*(floor_E+floor_T) mirrors the trainer's loss 0.5*(MSE_E+MSE_T) exactly — so it's directly comparable to the reported total val.

Why it's a good estimator (not a proxy)

Unlike nearest-neighbor / binning noise estimates, this directly samples the conditional distribution — same primary, same layout, many independent showers from the same generator + the
exact training kernel + the same log-transforms. So var_E is an (essentially unbiased, ddof=1) estimate of the true Var(y|q,xy), and the only approximations are statistical (finite M=64,
finite group count), not structural. Averaging over thousands of groups makes mean(var) stable.

The assumptions it leans on (all reasonable here):
1. Same label space — variance and σ both in log1p(E) / log1p(T·1e8), matching the trainer. ✓
2. Marginal σ is the trainer's z-score denominator. ✓ 
3. Group sampling ≈ token distribution — primaries from the generator's range, the 7 strategies uniformly, detectors uniformly; that's the distribution val averages over.
4. Same generator that made the corpus — the floor is a property of this data-generating process.

The one weak part: *_fired

floor_E_fired = var_E[fired].mean() / e_std**2 divides the fired-subset conditional variance by the global σ². That's a hybrid, not a proper conditional-on-fired floor — it answers "what
fraction of the global variance is fired-detector noise." Because the global σ is diluted by the ~62% always-empty detectors, the ratio can exceed 1 (that's why T_fired=1.06). A true
1−R²_fired floor would divide by the variance of y among fired groups, i.e. var of the fired labels, not e_std². So the all-detector floors are clean and directly comparable to val; the
_fired numbers are only a rough "fired detectors are very noisy" indicator and shouldn't be read as a fired-conditional MSE floor.

Want me to fix the _fired variant to normalize by fired-only variance (a real 1−R²_fired), and/or add a bootstrap CI over primaries so you have error bars on the floor?